# Job Sequencing

Dynex is the world’s only accessible neuromorphic quantum computing cloud for solving real-world problems, at scale.. This example demonstrates how to use the Dynex SDK to use Pyton to compute on the Dynex Platform with Python.

In [1]:
import dimod
from qubovert.problems import JobSequencing
from dynex import DynexConfig, ComputeBackend, BQM, DynexSampler, QPUModel

config = DynexConfig(compute_backend=ComputeBackend.QPU, qpu_model=QPUModel.APOLLO_RC1, use_notebook_output=True)



The goal of the JobSequencing problem is as follows. Given workers and jobs, where each job has a designated length, assign each of the jobs to one of the workers such that largest total length assigned to a worker is minimized.

In [2]:
job_lengths = {"job1": 2, "job2": 3, "job3": 1}
num_workers = 2
problem = JobSequencing(job_lengths, num_workers)
Q = problem.to_qubo()

In [3]:
bqm = dimod.BinaryQuadraticModel.from_qubo(Q.Q, Q.offset)

In [4]:
model = BQM(bqm)
sampler = DynexSampler(model, config=config)
sampleset = sampler.sample(num_reads=10, annealing_time=200)
print(sampleset)

INFO: [DYNEX-APOLLO-RC1] Timing breakdown:
INFO: [DYNEX-APOLLO-RC1]   Job upload:        0.25s
INFO: [DYNEX-APOLLO-RC1]   Time to 1st shot:  11.61s
INFO: [DYNEX-APOLLO-RC1]   Compute (Apollo):  11.61s
INFO: [DYNEX-APOLLO-RC1]   Solution download: 0.04s
INFO: [DYNEX-APOLLO-RC1]   Total elapsed:     11.90s
INFO: [DYNEX-APOLLO-RC1] Finished read after 5.432099999999984 seconds
INFO: [DYNEX-APOLLO-RC1] Sampleset ready with energy    0  1  2  3  4  5  6  7  8  9   energy num_oc.
0  1  0  0  1  1  0  0  0  0  0 0.140625       1
['BINARY', 1 rows, 1 samples, 10 variables]


   0  1  2  3  4  5  6  7  8  9   energy num_oc.
0  1  0  0  1  1  0  0  0  0  0 0.140625       1
['BINARY', 1 rows, 1 samples, 10 variables]


In [5]:
solution = problem.convert_solution(sampleset.first.sample)
obj = max(sum(job_lengths[i] for i in x) for x in solution) == 3
print('Optimal job sequence:', solution, 'correct?', obj)

Optimal job sequence: ({'job3', 'job1'}, {'job2'}) correct? True
